## ABT Preperation

At this point in our analysis we have stated a business decision, decided on a model or output, inspected the data, and created a cleaning plan. 

Throughout this section we will...
1) Create copies of the raw data so the original data remains available for comparison.
2) Controlled first-pass cleaning pipeline. Fix clear structural problems, protect the raw dataset, verify each major change, and document issues that require business context or human review.
3) Drop rows where mandatory fields are missing.
4) Convert Data Types and Clean Strings
5) Impute Missing Numeric Data
6) Validate the Cleaned Dataset
7) End Result and Interpretation
8) Create Cleaning Log explaining what we changed and why.
9) Create a data dictionary describing each column in the ABT.

By the end of this file we will have a preliminary ABT.

---
## Loading The Dataframe

The initial step in this project is to load the required libraries and the datasets. 
 
The libraries required are...   
        - pathlib : a file handler that treats paths as objects rather than text strings.   
        - pandas : a library of functions that are used to manipulate dataframes.  
        - numpy : a library of math functions.  

Next, we define the root folder where the project folders and files will be saved.  

The project directory contains the following folders...  
        - The data folder : This folder contains raw data files.  
        - The output folder : This folder contains the final ABT and processed data files.    
        - The report folder : This folder contains html reports, graphs, and analysis.  



In [179]:
# This code loads libraries and sets directories for the project.
#  pathlib is a file handler. treats paths as objects not strings
# pandas brings in df functions
# numpy provides math functions.

#Checking my Python environment 
import sys; print(sys.executable); print(sys.version)

from pathlib import Path
import pandas as pd
import numpy as np

#Set repository root to the current working directory. One step above readme location.

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

display(REPO_ROOT)

#Create directories for data, outputs, and reports if they don't exist.
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

#set max column display to 100 columns for pandas dataframes.
pd.set_option("display.max_columns", 100)

#This code loads the raw data into dataframes. And creates a column summary by table df.
# Load the datasets
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

# Create a dictionary (keys paired with values) to hold the DataFrames for easier access. We can loop through tables, etc.
tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

print("Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.")

/usr/bin/python3
3.13.5 (main, Aug 10 2026, 12:06:59) [GCC 14.2.0]


PosixPath('/workspaces/Project-Option-1-Matt-Flores')

Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.


---               
## Initial Cleaning Plan from Milestone 1

In this section we will complete the cleaning plan stated in Milsetone 1. 

The initial cleaning plan consists of the following steps:
1) Create copies of all raw data tables. All transformation will be done on copies. We don't want to change the raw data.
2) Log or create tables of any movieIds with missing or zero data.
3) Convert release_date to a date using pd.to_datetime(). Use errors = 'coerce' and log NaT.
4) Replace budget and revenue 0s with NaN.
5) Standardize the revenue and budget formats. Make them both float64.
6) Convert primary_genre, status, and original_language to category dtypes.
7) Convert vote_count, vote_average, and runtime 0 values to NaN.
8) Convert vote_count from float64 to Int64. It's a whole number.
9) Drop movies[adult]. All values are false. It provides no value to this analysis.
10) Drop movies that are missing all of the following: ratings in either table, genre attributes, or revenue. There are very few but best to remove them.
11) All movies dropped will be documented and placed in dropped dataset/list.
12) Parse through any lists included in the data sets and make a separate table (ex: movie_genres_parsed, keywords_parsed, cast_parsed)
13) Create a movies_ROI dataset that has movies with both revenue and budget.
14) Filter out any remaining suspicious values mentioned in profiling. Ex: run-time > 500 mins.

---
## Copying Raw Data 
Before we make any changes we must first create copies of the raw data. There are several reasons why creating copies of the raw data is important:
1) It preserves the original data for comparison purposes. 
2) It allows others to reproduce the analysis using the original data.
3) It let's us undo any changes made during cleanup and transformation.

In [180]:
# Use .copy() to create working dataframes
movies_clean = movies.copy() 
ratings_clean = ratings.copy() 
links_clean = links.copy() 
credits_clean = credits.copy() 
keywords_clean = movies.copy() 

print("All copies created!")

All copies created!


---
### Creating a cleaning log to record changes.

Before we start cleaning we will create an empty list to record the changes made throughout the cleaning and transformation process. After each change made to the data, we will record the method used, table and column impacted, a description of the change, reason, and notes.

In [181]:
cleaning_log = []

print("cleaning log created")

cleaning log created


---
## Cleaning - Movies
### Convert realease_date from object to datetime
Converting from object to datetime allows us to use pandas functions designed for timeseries data. It also converts invalid dates to NaT (not a time).

In [182]:
# Dtype of release date before conversion.
release_date_type_before = movies_clean['release_date'].dtype

# isna count of release date before conversion.
release_date_missing_before = movies_clean['release_date'].isna().sum()

# Convert using pd.to_datetime
movies_clean['release_date'] = pd.to_datetime(movies_clean['release_date'],  errors="coerce")

# Dtype of release date after conversion.
release_date_type_after = movies_clean['release_date'].dtype

# Dtype of release date after conversion.
release_date_missing_after = movies_clean['release_date'].isna().sum()

# Print isna count summary.
print(f"release_date converted to NaT: {release_date_missing_after - release_date_missing_before}")

# Print type before and after
print(f"release_date converted from {release_date_type_before} to {release_date_type_after}")

print(f"release dates span from {movies_clean['release_date'].min().date()} and {movies_clean['release_date'].max().date()}")

# Append changes to cleaning log
cleaning_log.append({
    'method': 'conversion',
    'table': 'movies',
    'column': 'release_date',
    'description': 'object to datetime64 (errors=coerce)',
    'reason': 'enable date filtering and pandas date features',
    'notes': 'no NaT. min date is 1902-09-01, max date is 2016-09-23',
})

release_date converted to NaT: 0
release_date converted from object to datetime64[ns]
release dates span from 1902-09-01 and 2016-09-23


### Convert vote_count from float64 to Int64

vote_counts are whole numbers. Converting to Int64 will show them as whole numbers and reveal errors on any vote_count value that has decimals.

In [183]:
# Dtype of release date before conversion.
vote_count_type_before = movies_clean['vote_count'].dtype

# isna count of release date before conversion.
vote_count_missing_before = movies_clean['vote_count'].isna().sum()

# Convert using Int64
movies_clean['vote_count'] = movies_clean['vote_count'].astype('Int64')

# Dtype of release date after conversion.
vote_count_type_after = movies_clean['vote_count'].dtype

# Dtype of release date after conversion.
vote_count_missing_after = movies_clean['vote_count'].isna().sum()

# Print isna count summary.
print(f"vote_count converted to NaN: {vote_count_missing_after - vote_count_missing_before}")

# Print type before and after
print(f"vote_count converted from {vote_count_type_before} to {vote_count_type_after}")

print(f"{(movies_clean['vote_count'] == 0).sum()} movies have no votes")

# Append changes to cleaning log
cleaning_log.append({
    'method': 'conversion',
    'table': 'movies',
    'column': 'vote_count',
    'description': 'float64 to Int64',
    'reason': 'show as whole number',
    'notes': 'no NaN. 44 movies have no votes',
})

vote_count converted to NaN: 0
vote_count converted from float64 to Int64
44 movies have no votes


### Convert budget to float64

Revenue is already set to float64. We will convert budget to float64 for consistency.

In [184]:
print(f"Revenue dtype is : {(movies_clean['revenue'].dtype)}")
print(f"Budget dtype before conversion is : {(movies_clean['budget'].dtype)}")

# Dtype of release date after conversion.
budget_type_before = movies_clean['budget'].dtype

# isna count of release date before conversion.
budget_missing_before = movies_clean['budget'].isna().sum()

# Convert using pd.to_datetime
movies_clean['budget'] = movies_clean['budget'].astype('float64')

# Dtype of release date after conversion.
budget_type_after = movies_clean['budget'].dtype

# Dtype of release date after conversion.
budget_missing_after = movies_clean['budget'].isna().sum()

# Print isna count summary.
print(f"budget converted to NaN: {budget_missing_after - budget_missing_before}")

# Print type before and after
print(f"budget converted from {budget_type_before} to {budget_type_after}")

budget_zero_count = (movies_clean['budget'] == 0)
print(f"{budget_zero_count.sum()} movies ({budget_zero_count.mean():.1%}) have 0 budget")

revenue_zero_count = (movies_clean['revenue'] == 0)
print(f"{revenue_zero_count.sum()} movies ({revenue_zero_count.mean():.1%}) have 0 revenue")

# Append changes to cleaning log
cleaning_log.append({
    'method': 'conversion',
    'table': 'movies',
    'column': 'budget',
    'description': 'int64 to float64',
    'reason': 'match revenue dtype',
    'notes': 'no NaN. 48.9% of movies have 0 budget',
})

Revenue dtype is : float64
Budget dtype before conversion is : int64
budget converted to NaN: 0
budget converted from int64 to float64
4443 movies (48.9%) have 0 budget
4322 movies (47.6%) have 0 revenue


### Convert primary_genre, status, and original_language to categories

The columns listed contain few unique values across thousands of rows. Converting them to categories establishes a fixed set of allowed values per column and allows us to create a custom order later if we'd like. It also reduces the memory and speed of calculations that use those columns. It's not necessary for this project but is good practice when working with large amounts of data.

The memory change shown below shows that converting to category reduced storage by 98%. When working with gigabytes of data this would be useful.

In [185]:
print("--- Category columns before conversion ---")
# List columns to convert to categories.
category_columns = ['primary_genre', 'status', 'original_language']

print(movies_clean[category_columns].dtypes)

# Column memory usage before converting
category_columns_memory_before = movies_clean[category_columns].memory_usage(deep=True).sum()

# Convert columns to category
movies_clean[category_columns] = movies_clean[category_columns].astype('category')

# Column memory usage after converting
category_columns_memory_after = movies_clean[category_columns].memory_usage(deep=True).sum()

print(f"\nMemory improvement from converting to category: {category_columns_memory_before/1e6:.2f} MB -> {category_columns_memory_after/1e6:.2f} MB")

print("\n--- Unique Values Check ---")
print(f"primary_genre has {movies_clean['primary_genre'].nunique()} unique values.")
print(f"status has {movies_clean['status'].nunique()} unique values.")
print(f"original_language has {movies_clean['original_language'].nunique()} unique values.")

print("\n--- primary_genre Values Check ---")
print(movies_clean["primary_genre"].value_counts().head(3))

print("\n--- status Values Check ---")
print(movies_clean["status"].value_counts().head(3))

# display(movies_clean.query("status == 'Rumored'"))

print("\n--- original_language Values Check ---")
print(movies_clean["original_language"].value_counts().head(3))

# Append changes to cleaning log
cleaning_log.extend([
    {'method': 'conversion', 'table': 'movies', 'column': 'primary_genre', 'description': 'object to category', 'reason': 'few unique values', 'notes': '20 unique values'},
    {'method': 'conversion', 'table': 'movies', 'column': 'status', 'description': 'object to category', 'reason': 'few unique values','notes': '4 unique values'},
    {'method': 'conversion', 'table': 'movies', 'column': 'original_language', 'description': 'object to category', 'reason': 'few unique values','notes': '42 unique values'},
])

--- Category columns before conversion ---
primary_genre        object
status               object
original_language    object
dtype: object

Memory improvement from converting to category: 1.48 MB -> 0.03 MB

--- Unique Values Check ---
primary_genre has 20 unique values.
status has 4 unique values.
original_language has 42 unique values.

--- primary_genre Values Check ---
primary_genre
Drama     2413
Comedy    2168
Action    1177
Name: count, dtype: int64

--- status Values Check ---
status
Released           9061
Rumored              11
Post Production       7
Name: count, dtype: int64

--- original_language Values Check ---
original_language
en    7947
fr     272
ja     185
Name: count, dtype: int64


### Create Derived Column: ROI

The plan for this analysis is to see which movie attributes are likely to result in a high probability of success. If revenue and budget are available we will use ROI as probability of success. Most movies must have revenue double their budget to reach profitability. We will assign roi and review summary information regarding the new column.

The formula for the derived column roi is: ***ROI*** = *(movies[revenue] - movies[budget]) / movies[budget]*

In [197]:
# Assign roi column
movies_clean = movies_clean.assign(roi = lambda x: (x["revenue"] -x["budget"]) / x["budget"] )

# Calculate roi description.
roi_median = movies_clean.query("budget > 0 and revenue > 0")['roi'].median()
roi_count = movies_clean.query("budget > 0 and revenue > 0")['roi'].shape[0]
roi_success = movies_clean.query("roi >= 1 and budget > 0 and revenue > 0")['roi'].shape[0]

print("roi derived column created")
print(f"median roi of movies with budget and rev > 0 is {roi_median:.2f}")
print(f"count of movies with budget and rev > 0 is {roi_count:,} ({roi_count/movies_clean.shape[0]*100:0.0f}% of all movies)  ")
print(f"count of movies with roi >=1 {roi_success:,} ({roi_success/roi_count*100:0.0f}% of all movies with roi calculated)  ")

print("\n--- describe() roi data ---")
display(movies_clean.query("budget > 0 and revenue > 0")['roi'].describe())

# Record roi in cleaning log
cleaning_log.append({
    'method': 'derivation',
    'table': 'movies',
    'column': 'roi',
    'description': 'roi = (rev-budget)/budget',
    'reason': 'model 1 output: roi >= 1 = successful',
    'notes': 'NaN or inf where budget or revenue missing',
})

roi derived column created
median roi of movies with budget and rev > 0 is 1.40
count of movies with budget and rev > 0 is 3,847 (42% of all movies)  
count of movies with roi >=1 2,211 (57% of all movies with roi calculated)  

--- describe() roi data ---


count    3.847000e+03
mean     5.707586e+03
std      2.428557e+05
min     -9.999981e-01
25%      8.539689e-02
50%      1.402459e+00
75%      3.695666e+00
max      1.239638e+07
Name: roi, dtype: float64

### Drop columns: adult and original_title

When cleaning data it's 

In [195]:
movies_clean

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,roi
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000.0,373554033.0,21.946943,7.7,5415,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,11.451801
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000.0,262797249.0,17.015539,6.9,2413,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.043035
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0.0,0.0,11.712900,6.5,92,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,NaN
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000.0,81452156.0,3.859495,6.1,34,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,4.090760
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0.0,76578911.0,8.387519,5.7,173,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9077,161944,159550,tt0255313,The Last Brickmaker in America,The Last Brickmaker in America,en,2001-09-23,85.0,8000000.0,0.0,0.038998,7.0,1,"[{'id': 18, 'name': 'Drama'}]","[{'name': 'Nasser Entertainment', 'id': 35802}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Drama,Drama,2001,-1.000000
9078,162542,392572,tt5165344,Rustom,रुस्तम,hi,2016-08-12,150.0,1000000.0,0.0,7.333139,7.3,25,"[{'id': 53, 'name': 'Thriller'}, {'id': 10749,...","[{'name': 'KriArj Entertainment', 'id': 91689}]","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Thriller|Romance,Thriller,2016,-1.000000
9079,162672,402672,tt3859980,Mohenjo Daro,Mohenjo Daro,hi,2016-08-11,155.0,15050000.0,16180000.0,1.423358,6.7,26,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'UTV Motion Pictures', 'id': 2320}, ...","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Adventure|Drama|History|Romance,Adventure,2016,0.075083
9080,163056,315011,tt4262980,Shin Godzilla,シン・ゴジラ,ja,2016-07-29,120.0,15000000.0,77000000.0,9.285519,6.6,152,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Cine Bazar', 'id': 5896}, {'name': ...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",False,Released,Action|Adventure|Drama|Horror|Science Fiction,Action,2016,4.133333


### Movie Conversion Review

Now that we converted the movie copy we can compare the raw movies data to the cleaned data and review the change log.

When comparing the raw vs clean data types we can see the 6 columns we converted. Everything looks accurate.

In [ ]:
# Build the DataFrame
movie_conversion_review = pd.DataFrame({
    'raw_dtype': movies.dtypes, 
    'clean_dtype': movies_clean.dtypes
})

print("--- Raw and clean dtypes ---")
display(movie_conversion_review.transpose())

print("\n--- cleaning log review ---")
display(pd.DataFrame(cleaning_log))

---               
## Data Structure

The first step in data profiling is reviewing the data structure. Data structure consists of rows and columns. To analyze data we must first understand what each row of the data represents and the attributes (columns) grouped with each row.

Instead of running shape() for each table I thought it would be best to aggregate each table's structure into summary table.  

I appended table name along with the associated columns into a single dataframe.  
I repeated the process a second time to gather the shape info from each table.

In [ ]:
# Create a blank list called column_summary that we can populate.
column_summary = []

# Populate the list using a loop that appends table names and each column within that table.
for name, df in tables.items():
    column_summary.append({"table_name": name, **{f"col_{i+1}": col for i, col in enumerate(df.columns)}})

# Build a summary DataFrame that shows the table name and each column name in separate columns.
# Converted summary_data list into a pd.dataframe and set index equal to table name.
column_summary_df = pd.DataFrame(column_summary).set_index("table_name")

print("--- Column Summary ---")
# Display the result
display(column_summary_df)

raw_data_shape = []
# Print out each df and it's shape. For is a loop function. "for every name in tables dictionary, print the shape variables"
for name, df in tables.items():
     raw_data_shape.append({"table_name": name, "rows": df.shape[0], "columns": df.shape[1]})

raw_data_shape_df = pd.DataFrame(raw_data_shape).set_index("table_name")

print("--- Table Shapes ---")
# Display the results
display(raw_data_shape_df)

# Audit keys to make sure there are no duplicates or blanks.
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

print("--- Key Audit ---")
display(key_audit)

We can confirm a few facts:  
1) **Granularity Match**: The movies, credits, and keywords datasets contain 9,082 rows. The links df contains 43 more rows. It's possible the links table contains additional movieids that are not included in the other tables.

2) **Ratings Shape**: The ratings df has almost 100k rows. The most popular movies might receive a large portion of these reviews. This will require further analysis.  

3) **Aggregation Required**: Since the ratings row level doesn't match the other tables any joining of the ratings table will require aggregation. We could examine the average rating per movie, distribution of review scores, count of reviews, etc.  

4) **Movie Sales Figures**: The movies dataset contains budget and revenue columns. These will be useful when calculating ROI per movie and sorting by revenue. One thing to consider is highly rated movies might not make the most revenue. Some classic movies such as "Fight Club" and "Donnie Darko" had a low or negative ROI.

5) **Key Audit**: The audit was passed. There are no duplicate keys in any table.

---               
## Initial Inspection

This step is exploratory. We know the dataset structure, but it's helpful to see a snippet of each table. Let's loop through each table and output head(3) and tail(3)

In [ ]:
# Loop through dataframes and display head(3) tail(3)
for name, df in tables.items():
    print(f'\n{name}: head and tail')
    display(pd.concat([df.head(3),df.tail(3)]))

### Head & Tail Results
The head and tail output resulted in some key findings:  
    - genres, production_companies, and production_countries are lists.  
    - genre list is genres reformatted as a string list delimited by "|".  
    - I'm not sure yet how ratings[timestamp] will be used. It may be redundant for this analysis.  
    - top_cast is a string list delimited by "|".  
    - keywords_list appears very detailed. Also a string separated by "|".  

---

Now that we've reviewed the head and tail. We'll loop through the describe function.

In [ ]:
# For every name in tables dictionary print a title and the describe results
for name, df in tables.items():
    print((f'{name}: Describe Summary'))
    display(df.describe().T)

### Describe Function Results

The describe tables are easy to read and straight forward, but there are a few things I'd like to call out:

**movies**:   
    - Zero values exist in runtime, budget, and revenue. These are most likely missing values.  
    - There are extremely long runtimes. The max runtime is 1,140 mins. These will need to be cleaned from the data.  
    - Vote scoring system is from 1 to 10. The avg rating is 6.4.  
    - Min release year is 1902. Max year is 2016. Older movies may be outside the scope of this project.

**ratings**:  
    - The rating scale is from .5 to 5.  
    - The reviews occurred between 1995 and 2016. The movies reviewed could have been released prior to 1995.

***links, credits, and keywords data have no numeric data to describe.***   


---               
## Inspecting Missing, Duplicate, & Unusual Values

Next, we will profile each dataset and document issues that may require cleaning in the next project milestone. There are 5 sections, each dedicated to a dataset. 

For each dataset we will:  
    - Review data types.  
    - Check for missing values, empty strings, and zero values.  
    - Ensure no primary key is duplicated.  
    - Note suspicious, impossible, or confusing values


---
### Movies

The movies dataset contains over 9000 movies with 21 columns of details. We will profile the data below using python functions such as .dtypes, nunique, .isna.mean, and .isna.sum. It's more clear if we can view all the details regarding a dataset in a single table output.

In [ ]:
# Create a dataframe to store movies data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.

# Convert all number columns into true or false so we can count where 0 is true
num_cols = movies.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = movies.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# Build the DataFrame
movies_profile = pd.DataFrame({
    'dtype': movies.dtypes, 
    'unique_values' : movies.nunique(),
    'duplicate_values' : movies.count() - movies.nunique(),
    'missing_rate': round(movies.isna().mean() * 100,2), 
    'missing_count': movies.isna().sum(),
    'zero_count': zero_mask.sum().reindex(movies.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(movies.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(movies.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(movies.columns, fill_value=0) * 100,2)
})

print("Movies Profile Table")
display(movies_profile.sort_values(by='missing_rate', ascending=False))


The output table generated above provides some key profiling information:

1) **dtypes** : There are a few things that could be changed.
    - ***release_date*** is an object. It should be converted to a date using pd.to_datetime().
    - ***vote_count*** is set as float64 but it's a whole number, so it should be Int64.
    - ***revenue*** and ***budget*** represent dollars but are different dtypes. Let's plan on standardizing them as float64. Float64 allows for NaN values, which will prevent them from being included in math calculations.
    - (***primary_genre, status, original_language***) contain few unique values. They could be converted to ***category*** data type. Doing so will improve grouping and sorting processing speeds.  

2) **Missing Values** : The output table display missing count/rate, zero count/rate, and empty count/rate.
    - Missing count/rate are calculated using isna mean and isna sum. Those functions identify null or missing values. The data appears to be *complete* in regards to null and missing.
    - The zero count/rate reveal missing information that will impact the analysis. *Almost 50% of the revenue/budget values are 0*. This means when conducting analysis on revenue/budget we must omit these movieIds. Including them in any equations will skew the results.
    - empty count/rate shows 0 because there are no blank strings or values. There are only nulls/isna or 0.

3) **Duplicates** : The movies dataset has 9,082 rows.
    - ***movieId, tmdbId, and imdb_id*** have 9,082 unique values and rows. This makes logical sense and proves that the row level data represents a single movie.
    - It's possible to have duplicate movie ***titles***. The data spans from 1902 to 2016. Over almost 100 years there have been 273 movies with a title that's already been used.
    - Any column that represents a group or label is allowed to have duplicates. These are: ***genres_list, primary_genre, status, original language, genres, adult, production countries, production companies, and release year***.
    - It's suspicious that ***revenue*** and ***budget*** have duplicates, but remember about 50% of them are 0. So 50% of them should be duplicates. Also consider that the dollars amounts have low granularity. The budget/revenue figures are a rough ball park of the real amounts. Which reduces the variety of the data.
    - ***vote_average*** is based on a scale of 1-10. It makes sense that there are duplicates.

---
### Ratings

The ratings dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [ ]:
# Create a dataframe to store ratings data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.

# Convert all number columns into true or false so we can count where 0 is true
num_cols = ratings.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = ratings.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# Build the DataFrame
ratings_profile = pd.DataFrame({
    'dtype': ratings.dtypes, 
    'unique_values' : ratings.nunique(),
    'duplicate_values' : ratings.count() - ratings.nunique(),
    'missing_rate': round(ratings.isna().mean() * 100,2), 
    'missing_count': ratings.isna().sum(),
    'zero_count': zero_mask.sum().reindex(ratings.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(ratings.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(ratings.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(ratings.columns, fill_value=0) * 100,2)
})

print("ratings Profile Table")
display(ratings_profile.sort_values(by='missing_rate', ascending=False))

- One interesting thing to note in the output is there are only 671 unique userIds. It is possible that 671 users created 99,139 reviews. We will assume that's accurate and all reviews are legitimate.
- There are 10 unique values for rating. This is because the rating scale is in increments of .5 from .5 to 5.

---
### Links

The links dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [ ]:
# Create a dataframe to store links data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.

# Convert all number columns into true or false so we can count where 0 is true
num_cols = links.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = links.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# Build the DataFrame
links_profile = pd.DataFrame({
    'dtype': links.dtypes, 
    'unique_values' : links.nunique(),
    'duplicate_values' : links.count()- links.nunique(),
    'missing_rate': round(links.isna().mean() * 100,2), 
    'missing_count': links.isna().sum(),
    'zero_count': zero_mask.sum().reindex(links.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(links.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(links.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(links.columns, fill_value=0) * 100,2)
})

print("links Profile Table")
display(links_profile.sort_values(by='missing_rate', ascending=False))

- The only thing worth noting is there are 13 missing the movie database ids. We wont be able to cross reference those 13 movies using the tmdbId since they don't exist.

---
### Credits

The credits dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [ ]:
# Create a dataframe to store credits data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.

# Convert all number columns into true or false so we can count where 0 is true
num_cols = credits.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = credits.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# Build the DataFrame
credits_profile = pd.DataFrame({
    'dtype': credits.dtypes, 
    'unique_values' : credits.nunique(),
    'duplicate_values' : credits.count() - credits.nunique(),
    'missing_rate': round(credits.isna().mean() * 100,2), 
    'missing_count': credits.isna().sum(),
    'zero_count': zero_mask.sum().reindex(credits.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(credits.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(credits.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(credits.columns, fill_value=0) * 100,2)
})

print("credits Profile Table")
display(credits_profile.sort_values(by='missing_rate', ascending=False))

In [ ]:
# Return the top 10 most frequent values in the director and top_cast columns
display(credits['director'].value_counts().head(10))
display(credits['top_cast'].value_counts().head(10))

The missing rate is insignificant in the credits table. I pulled the value_counts to review the top duplicates from the director and top_casts tables.

We can see that the directors and actors can have dozens of movies associated with them.

---
### Keywords

The keywords dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [ ]:
# Create a dataframe to store keywords data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.

# Convert all number columns into true or false so we can count where 0 is true
num_cols = keywords.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = keywords.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# Build the DataFrame
keywords_profile = pd.DataFrame({
    'dtype': keywords.dtypes, 
    'unique_values' : keywords.nunique(),
    'duplicate_values' : keywords.count() - keywords.nunique(),
    'missing_rate': round(keywords.isna().mean() * 100,2), 
    'missing_count': keywords.isna().sum(),
    'zero_count': zero_mask.sum().reindex(keywords.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(keywords.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(keywords.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(keywords.columns, fill_value=0) * 100,2)
})

print("keywords Profile Table")
display(keywords_profile.sort_values(by='missing_rate', ascending=False))

The missing rate is about 8% in the keywords table. If these don't get filtered out in cleaning steps they may have to be removed directly or flagged. 8% is a substantial amount of key data to drop.

---               
## Initial Cleaning Plan

The initial cleaning plan will consist of the following steps:
1) Create copies of all raw data tables. All transformation will be done on copies. We don't want to change the raw data.
2) Log or create tables of any movieIds with missing or zero data.
3) Convert release_date to a date using pd.to_datetime(). Use errors = 'coerce' and log NaT.
4) Replace budget and revenue 0s with NaN.
5) Standardize the revenue and budget formats. Make them both float64.
6) Convert primary_genre, status, and original_language to category dtypes.
7) Convert vote_count, vote_average, and runtime 0 values to NaN.
8) Convert vote_count from float64 to Int64. It's a whole number.
9) Drop movies[adult]. All values are false. It provides no value to this analysis.
10) Drop movies that are missing all of the following: ratings in either table, genre attributes, or revenue. There are very few but best to remove them.
11) All movies dropped will be documented and placed in dropped dataset/list.
12) Parse through any lists included in the data sets and make a separate table (ex: movie_genres_parsed, keywords_parsed, cast_parsed)
13) Create a movies_ROI dataset that has movies with both revenue and budget.
14) Filter out any remaining suspicious values mentioned in profiling. Ex: run-time > 500 mins.

---               
## Potential target or outcome variable

*See "The Model" section at the top of the notebook.*

Outcome variables will be: 
1) Model 1: roi_threshold_met = True/False
2) Model 2: rating_threshold_met = True/False


---               
## Early ethics, privacy, governance, or responsible-use concerns

**Confidentiality**: This analysis is confidential since it will determine the content pipeline for Netflix. Only executive and management teams will have access to the analysis results. If competitors know which movies are in our pipeline they might create similar movies with the same themes.

**Financial Stewardship**: We have a duty to manage our $10B budget wisely. We can't make 1 high budget action movie. Our movie selection must be diverse. Lower budgets + high success score attributes will allow us to make a variety of content for our customers.

**Governance**: The analyst team is responsible for governing this data and only sharing it with the appropriate business teams.

**Privacy**: All analysis and data will remain internal. The review data contains userId. Using the scores and timestamp, userIds could be identified indirectly. Rating history can reveal personal information. It's best to share aggregate review data so it's not possible to identify an individual.  
Reference: https://privacylaw.proskauer.com/2009/12/articles/invasion-of-privacy/netflix-sued-for-largest-voluntary-privacy-breach-to-date/


---

In [ ]:
# Various validation checks.

# Query to make sure older movies have revenue/budget info.
display(movies.query("revenue > 0 and budget >0 and release_year < 1995"))

# Check Donnie Darko roi
display(movies.query("title == 'Donnie Darko'"))

# Print ratings columns. refresh memory.
print(ratings.columns)

#Check to see if ratings table has ratings for movies prior to the first ratings date in 1995. 
# The output shows ratings data exists for movies released before 1995
display(
    ratings[ratings['movieId'].isin(movies.query('release_year < 1995')['movieId'])]
    .merge(movies[['movieId', 'title', 'release_date']], on='movieId')
    [['title', 'release_date', 'rating_datetime']]
    .head(20)
)

# movies with vote_average == 0 and votes exist.
movies.query('vote_average == 0 and vote_count > 0')[['title', 'vote_count', 'vote_average']]

# check what movies are in links but not in movies.
display(links[~links['movieId'].isin(movies['movieId'])])
(~links['movieId'].isin(movies['movieId'])).sum()

# check what movies are in ratings but not in movies
(~ratings['movieId'].isin(movies['movieId'])).sum()